# LA 07 — W01: anatomy of the A03b 1:1 trades (winners vs losers), all data layers

Protocol: `discussions/Liquidity_Absorption/027_claude_w01_winner_anatomy_preregistration.md`. Engine `liquidity_absorption_w01.py`.

Needs the EXT outputs (`EXT/03b`, `EXT/03d`) and `DERIBIT_DATA_ROOT/history`. The per-minute panel (~1M rows) is written to `DERIBIT_DATA_ROOT/la_w01/`, not to the repo. `LAW01_SYNTHETIC=1` for a smoke run.

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src' / 'data_engineering').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / 'src'))
from src.data_engineering.modeling import liquidity_absorption_w01 as w
SYN = os.getenv('LAW01_SYNTHETIC', '').strip() == '1'
if SYN:
    TMP = Path(tempfile.mkdtemp(prefix='law01_')); EXT = TMP / 'EXT'; HIST = TMP / 'history'; LOCAL = TMP / 'local'; OUT = TMP / '07'
    w.synthetic_ext(EXT, HIST, n=160)
else:
    DATA = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    HIST = DATA / 'history'; LOCAL = DATA / 'la_w01'
    EXT = REPO / 'notebooks' / 'Liquidity_Absorption' / 'EXT'; OUT = REPO / 'notebooks' / 'Liquidity_Absorption' / '07'
OUT.mkdir(parents=True, exist_ok=True)
(OUT / '07_registration.json').write_text(json.dumps(w.registration(), indent=1, default=str))
print(w.VERSION, '| synthetic' if SYN else '', '| EXT', EXT, '| local', LOCAL, '| out', OUT)

## 1. Build: trade features + per-minute panel (−240…+240) from Deribit prints, A06 context and the 18:00 ET profile

Cached: delete `LOCAL/W01_panel.parquet` to rebuild.

In [ ]:
t0 = time.time()
if (LOCAL / 'W01_panel.parquet').exists() and (LOCAL / 'W01_trade_features_raw.parquet').exists() and not SYN:
    F = pd.read_parquet(LOCAL / 'W01_trade_features_raw.parquet'); P = pd.read_parquet(LOCAL / 'W01_panel.parquet')
else:
    F, P = w.build(EXT, HIST, LOCAL)
print(F.shape, P.shape, f'{time.time()-t0:.0f}s')
display(F.groupby(['split', 'reason']).size().unstack(fill_value=0))

## 2. Analysis: H1 (CVD toward target at landmarks), management rules, bursts, feature screen

In [ ]:
t0 = time.time()
DEC = w.run_analysis(F, P, OUT)
print(f'{time.time()-t0:.0f}s'); print(json.dumps(DEC, indent=1, default=str))

In [ ]:
H = pd.read_csv(OUT / 'W01_h1_cvd_landmark.csv'); display(H.round(3))
RU = pd.read_csv(OUT / 'W01_management_rules.csv'); display(RU.round(3))
BS = pd.read_csv(OUT / 'W01_burst_summary.csv'); display(BS.round(3))

In [ ]:
S = pd.read_csv(OUT / 'W01_feature_screen.csv')
display(S[S.survivor].round(4))
display(S.sort_values('p').groupby('family').head(15)[['family','feature','n_target','n_stop','med_target','med_stop','rank_biserial','rho_with_y','p','q']].round(4))

## 3. Pictures: what winners and losers look like (MINING split only)

In [ ]:
E = pd.read_csv(OUT / 'W01_curves_entry.csv.gz'); XA = pd.read_csv(OUT / 'W01_curves_exit.csv')
fig, ax = plt.subplots(2, 3, figsize=(20, 9))
for j, col in enumerate(['px_R', 'cvd_from_entry', 'f3']):
    for oc, c in (('TARGET', 'tab:green'), ('STOP', 'tab:red'), ('TIMEOUT', 'tab:grey')):
        e = E[(E.split == 'MINING') & (E.reason == oc)]
        ax[0, j].plot(e.rel, e[col], c=c, label=oc)
        x = XA[(XA.split == 'MINING') & (XA.reason == oc)]
        if len(x): ax[1, j].plot(x.to_exit, x[col], c=c, label=oc)
    ax[0, j].axvline(0, c='k', lw=.6); ax[0, j].set_title(f'{col}, median, entry-aligned (minutes)'); ax[0, j].legend()
    ax[1, j].set_title(f'{col}, median, last 60 min before exit'); ax[1, j].legend()
    for a in ax[:, j]: a.axhline(0, c='k', lw=.4); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / 'W01_curves.png', dpi=110); plt.show()

In [ ]:
B = pd.read_csv(OUT / 'W01_burst_per_trade.csv.gz')
b = B[B.had_burst & B.split.eq('MINING') & B.reason.isin(['TARGET', 'STOP'])]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for oc, c in (('TARGET', 'tab:green'), ('STOP', 'tab:red')):
    g = b[b.reason == oc]
    ax[0].hist(g.prog_at_onset_R.clip(-1.2, 1.2), bins=40, alpha=.5, color=c, label=oc, density=True)
    ax[1].hist(g.onset_min.clip(0, 240), bins=48, alpha=.5, color=c, label=oc, density=True)
ax[0].set_title('progress toward target (R) when the first burst toward the target completes'); ax[0].legend()
ax[1].set_title('minutes from entry to the first burst'); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / 'W01_bursts.png', dpi=110); plt.show()